# Build an Excel report from 140 MB of IMF data: fabric-rlm vs a direct LLM call

One task, tried two ways. The job: from the IMF's Consumer Price Index data,
build a formatted Excel workbook with a pivot of the 10 highest-inflation
countries by year, a merged title cell, styled headers, and a second sheet
listing every qualifying country.

The data comes straight from the IMF's public SDMX API (the dataset page is
https://data.imf.org/en/datasets/IMF.STA:CPI). The pull used here is about
140 MB of CSV: 1.57 million observation rows by 22 columns, roughly 35 million
values. As text that is on the order of 36 million tokens, about ninety times
the 400,000-token context window of gpt-5.1. No prompt can hold it.

First we ask gpt-5.1 directly, with as much of the raw file as fits in its
prompt. Then we hand the same question to gpt-5-mini, a model about 5x
cheaper, running through fabric-rlm so it can query the file with code. A
DuckDB query computes the ground truth and grades the result cell by cell.

Before running: attach a Lakehouse to this notebook (the data file and the
generated reports are written to Lakehouse Files).

In [ ]:
%pip install -q fabric-rlm[analytics] openpyxl

## Download the data

One GET against the IMF SDMX 3.0 API. No API key needed. The IMF server
usually delivers the 140 MB in under a minute. The cell skips the download if
the file is already in the Lakehouse.

In [ ]:
import os, urllib.request

DATA_DIR = "/lakehouse/default/Files"
DATA_PATH = os.path.join(DATA_DIR, "imf_cpi.csv")
URL = (
    "https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.STA/CPI/5.0.0/*.*.*.*.*"
    "?c%5BTIME_PERIOD%5D=ge:2017-01-01+le:2026-12-31"
)

if not os.path.exists(DATA_PATH):
    req = urllib.request.Request(URL, headers={"Accept": "application/vnd.sdmx.data+csv"})
    with urllib.request.urlopen(req, timeout=900) as r, open(DATA_PATH, "wb") as fh:
        while chunk := r.read(1 << 20):
            fh.write(chunk)
print(f"{os.path.getsize(DATA_PATH) / 1e6:.0f} MB at {DATA_PATH}")

## Pick the models

`FabricLM` uses the capacity's built-in Azure OpenAI endpoint, so there is no
key to manage and nothing to provision. The hosted model list is at
https://learn.microsoft.com/en-us/fabric/data-science/ai-services/ai-services-overview#consumption-rate

In [ ]:
from fabric_rlm import FabricLM

lm_big = FabricLM("gpt-5.1")
lm_mini = FabricLM("gpt-5-mini")

## The task

In [ ]:
REPORT_PATH = os.path.join(DATA_DIR, "cpi_report.xlsx")

TASK = """You are given IMF CPI data in SDMX-CSV format (one observation per row)
at data_file. Build an Excel report at report_path (create the file with openpyxl).

Data selection: rows with INDEX_TYPE='CPI', COICOP_1999='_T',
TYPE_OF_TRANSFORMATION='YOY_PCH_PA_PT', FREQUENCY='M'. TIME_PERIOD looks like
'2021-M01'; COUNTRY is an ISO3 code; the value is in OBS_VALUE. A qualifying
country has all 60 monthly observations for 2021 through 2025.

The workbook has two sheets.

Sheet 'Report':
- A1:G1 merged, containing exactly: Average year-over-year CPI inflation (%), 2021-2025
- Row 2 headers, bold, with a solid light-gray fill (PatternFill fgColor D9D9D9):
  Country, 2021, 2022, 2023, 2024, 2025, Avg 2021-2025
- Set column A width to 32.
- Rows 3 to 12: the 10 qualifying countries with the highest five-year average of
  their monthly YoY values, sorted descending by that average. Each year column is
  that calendar year's average of the 12 monthly values; 'Avg 2021-2025' is the
  average of all 60 monthly values. Round to 2 decimals and apply number format
  '0.00' to every numeric cell.
- Row 13: column A = 'Median (all qualifying countries)', column G = the median
  across ALL qualifying countries of their five-year averages, 2 decimals,
  number format '0.00'.

Sheet 'All countries':
- Row 1 headers, bold: Country, Avg 2021-2025
- One row per qualifying country, sorted descending by the five-year average,
  values rounded to 2 decimals with number format '0.00'.

Save to report_path. Then reload the saved file with openpyxl and verify EVERY
requirement above: both sheet names exactly as given, the merge, bold headers
with the D9D9D9 fill, the column width, the '0.00' number formats, the sorted
rows, and the 'All countries' row count. Fix and re-save until every check
passes. Only then SUBMIT with n_countries (the count of qualifying countries)
and median_avg (the row-13 median value)."""

## Ask gpt-5.1 directly

200,000 characters of the file go into the prompt, which is under 0.15 percent
of it, and the model cannot create a file anyway, so it only has to report the
numbers that would go into the workbook. When we ran this it spent about
109,000 prompt tokens and then answered, correctly, that the question cannot
be answered from the slice it was shown: the visible rows do not even include
the required series.

In [ ]:
import time

PLAIN_PROMPT = """The text below is the beginning of a large CSV of IMF CPI data in
SDMX-CSV format (one observation per row). The full file is about 140 MB; only
this slice fits in your context.

Rows with INDEX_TYPE='CPI', COICOP_1999='_T', TYPE_OF_TRANSFORMATION='YOY_PCH_PA_PT',
FREQUENCY='M' are monthly year-over-year all-items CPI inflation. TIME_PERIOD looks
like '2021-M01'; COUNTRY is an ISO3 code; the value is in OBS_VALUE. A qualifying
country has all 60 monthly observations for 2021 through 2025.

Report, as JSON only:
{"n_countries": <count of qualifying countries>,
 "top10": [[country, avg2021, avg2022, avg2023, avg2024, avg2025, avg_5yr], ...],
 "median_avg": <median across all qualifying countries of their 5-year averages>}
top10 = the 10 qualifying countries with the highest 5-year average of monthly YoY
values, sorted descending, yearly values = that year's average of 12 monthly values,
all numbers to 2 decimals."""

head = open(DATA_PATH, encoding="utf-8").read(200_000)
t0 = time.time()
plain_text = lm_big(f"{PLAIN_PROMPT}\n\n--- FILE SLICE ---\n{head}")[0]
plain_seconds = time.time() - t0
plain_usage = lm_big.history[-1]["usage"]
print(plain_text[:1500])

## Same question, through fabric-rlm

Now gpt-5-mini gets the file as an input instead of pasted text. It writes
DuckDB and openpyxl code in the subprocess: aggregate 1.57 million rows, pivot
by year, merge and style the header, write both sheets, save the workbook.

In [ ]:
from fabric_rlm import File, RLM

t0 = time.time()
rlm = RLM.task(
    task=TASK,
    inputs={"data_file": File(DATA_PATH), "report_path": REPORT_PATH},
    outputs=["n_countries", "median_avg"],
    lm=lm_mini,
    skills=["data_exploration", "excel_modify"],
    max_turns=10,
    timeout=600.0,
)
result = rlm.run()
rlm_seconds = time.time() - t0
result.payload

## Ground truth

A DuckDB pivot computes the true report from the same file.

In [ ]:
import duckdb, statistics

con = duckdb.connect()
rows = con.execute(f"""
WITH obs AS (
    SELECT COUNTRY, substr(TIME_PERIOD, 1, 4) AS yr, OBS_VALUE
    FROM read_csv_auto('{DATA_PATH}')
    WHERE INDEX_TYPE = 'CPI' AND COICOP_1999 = '_T'
      AND TYPE_OF_TRANSFORMATION = 'YOY_PCH_PA_PT' AND FREQUENCY = 'M'
      AND substr(TIME_PERIOD, 1, 4) BETWEEN '2021' AND '2025'
      AND OBS_VALUE IS NOT NULL
), complete AS (
    SELECT COUNTRY FROM obs GROUP BY COUNTRY HAVING count(*) = 60
), yearly AS (
    SELECT o.COUNTRY, yr, avg(OBS_VALUE) AS y_avg
    FROM obs o JOIN complete c USING (COUNTRY) GROUP BY o.COUNTRY, yr
), fivey AS (
    SELECT o.COUNTRY, avg(OBS_VALUE) AS avg5
    FROM obs o JOIN complete c USING (COUNTRY) GROUP BY o.COUNTRY
)
SELECT f.COUNTRY,
  max(CASE WHEN yr = '2021' THEN y_avg END), max(CASE WHEN yr = '2022' THEN y_avg END),
  max(CASE WHEN yr = '2023' THEN y_avg END), max(CASE WHEN yr = '2024' THEN y_avg END),
  max(CASE WHEN yr = '2025' THEN y_avg END), max(avg5)
FROM fivey f JOIN yearly y ON f.COUNTRY = y.COUNTRY
GROUP BY f.COUNTRY ORDER BY max(avg5) DESC
""").fetchall()

truth = {
    "n_countries": len(rows),
    "top10": [[r[0]] + [round(v, 2) for v in r[1:]] for r in rows[:10]],
    "median_avg": round(statistics.median(r[6] for r in rows), 2),
}
truth["top10"]

## Grade the workbook

Reload the workbook with openpyxl and check it structurally (merged title,
bold and filled headers, column width, number formats, both sheets) and
numerically (all 60 pivot values, the median, and the full country list,
within 0.02). The direct call is graded generously: it only has to name the
ten correct countries anywhere in its answer.

In [ ]:
from openpyxl import load_workbook

# OpenAI list prices, USD per 1M tokens (input, output), for the cost column.
PRICE = {"gpt-5.1": (1.25, 10.00), "gpt-5-mini": (0.25, 2.00)}

def cost(model, prompt_tokens, completion_tokens):
    p_in, p_out = PRICE[model]
    return (prompt_tokens * p_in + completion_tokens * p_out) / 1e6

def ok(a, b):
    return abs(float(a) - b) <= 0.02

wb = load_workbook(REPORT_PATH)
ws = wb["Report"]
allc = wb["All countries"]
top = truth["top10"]

checks = {
    "A1_G1_merged": "A1:G1" in [str(r) for r in ws.merged_cells.ranges],
    "title": ws["A1"].value == "Average year-over-year CPI inflation (%), 2021-2025",
    "headers_bold": all(ws.cell(row=2, column=c).font.bold for c in range(1, 8)),
    "header_fill": str(ws["A2"].fill.fgColor.rgb).endswith("D9D9D9"),
    "col_A_width": ws.column_dimensions["A"].width >= 30,
    "number_format": ws["B3"].number_format == "0.00",
    "top10_countries": [ws.cell(row=3 + i, column=1).value for i in range(10)] == [r[0] for r in top],
    "top10_values": all(ok(ws.cell(row=3 + i, column=2 + j).value, top[i][1 + j]) for i in range(10) for j in range(6)),
    "median_row": ok(ws["G13"].value, truth["median_avg"]),
    "all_countries_rows": allc.max_row == truth["n_countries"] + 1,
    "all_countries_sorted": allc["A2"].value == top[0][0],
}
checks

In [ ]:
plain_correct = all(r[0] in plain_text for r in truth["top10"])
plain_tokens = plain_usage["prompt_tokens"] + plain_usage["completion_tokens"]
plain_cost = cost("gpt-5.1", plain_usage["prompt_tokens"], plain_usage["completion_tokens"])

rlm_correct = all(checks.values())
rlm_tokens = result.total_prompt_tokens + result.total_completion_tokens
rlm_cost = cost("gpt-5-mini", result.total_prompt_tokens, result.total_completion_tokens)

plain_result = "passed" if plain_correct else "failed"
rlm_result = "passed" if rlm_correct else "failed"
print(f"{'run':<24} {'result':<8} {'workbook':<18} {'tokens':>9} {'cost':>8} {'seconds':>9}")
print(f"{'direct call, gpt-5.1':<24} {plain_result:<8} {'none':<18} {plain_tokens:>9,} {plain_cost:>7.3f}$ {plain_seconds:>9.1f}")
print(f"{'fabric-rlm, gpt-5-mini':<24} {rlm_result:<8} {'built and graded':<18} {rlm_tokens:>9,} {rlm_cost:>7.3f}$ {rlm_seconds:>9.1f}")

## A harder task: longest high-inflation streaks

How far can the cheap model go? This one is genuinely fiddly: find each
country's longest unbroken run of months at or above 10 percent inflation,
with two levels of tie-breaks, the peak within each streak, a two-color
conditional-formatting scale, and a bar chart embedded in the sheet. In our
run gpt-5-mini cleared every check in 6 turns for about four cents.

In [ ]:
REPORT2_PATH = os.path.join(DATA_DIR, "streaks_report.xlsx")

STREAKS_TASK = """You are given IMF CPI data in SDMX-CSV format (one observation per row)
at data_file. Build an Excel report at report_path (create the file with openpyxl).

Data selection: rows with INDEX_TYPE='CPI', COICOP_1999='_T',
TYPE_OF_TRANSFORMATION='YOY_PCH_PA_PT', FREQUENCY='M'. TIME_PERIOD looks like
'2021-M01'; COUNTRY is an ISO3 code; the value is in OBS_VALUE. Consider months
2021-M01 through 2025-M12 and only countries with all 60 monthly observations.

Definitions:
- A high-inflation streak is a MAXIMAL run of consecutive months whose value is
  greater than or equal to 10.0.
- For each country that has at least one streak, take its LONGEST streak; if
  several streaks tie on length, take the one with the earliest start month.
- The streak's peak is the maximum value inside that streak, rounded to 2
  decimals; the peak month is the earliest month attaining that maximum.

The workbook has two sheets.

Sheet 'Streaks':
- A1:F1 merged, containing exactly: Longest high-inflation streaks (YoY >= 10%), 2021-2025
- Row 2 headers, bold, with a solid light-gray fill (PatternFill fgColor D9D9D9):
  Country, Months, Start, End, Peak YoY, Peak month
- Set the width of columns A and F to 14.
- Rows 3 to 17: the top 15 countries, sorted by streak length descending, then
  start month ascending, then country code ascending. Months is an integer;
  Start, End, Peak month are period strings like '2021-M01'; Peak YoY is numeric
  with number format '0.00'.
- Apply a 2-color conditional formatting color scale to E3:E17 (the Peak YoY
  column), from white (FFFFFF) at the minimum to red (FF0000) at the maximum.
- Add a bar chart anchored at H2 titled 'Peak YoY, top 10' showing the Peak YoY
  values of rows 3 to 12, with the country codes of those rows as categories.

Sheet 'Summary':
- A1: Countries analyzed         B1: <count of qualifying countries>
- A2: Countries with a streak    B2: <count of countries with at least one streak>
- A3: Longest streak country     B3: <country in row 3 of the Streaks sheet>
- A4: Longest streak months      B4: <its streak length>

Save to report_path. Then reload the saved file with openpyxl and verify EVERY
requirement above: sheet names, the merge, bold headers with the D9D9D9 fill,
the column widths, the '0.00' number format, the exact sort order including both
tie-breaks, the Summary values, the conditional formatting rule on E3:E17, and
that the sheet contains exactly one chart. Fix and re-save until every check
passes. Only then SUBMIT with n_analyzed, n_with_streak, longest_country,
longest_len."""

In [ ]:
t0 = time.time()
rlm2 = RLM.task(
    task=STREAKS_TASK,
    inputs={"data_file": File(DATA_PATH), "report_path": REPORT2_PATH},
    outputs=["n_analyzed", "n_with_streak", "longest_country", "longest_len"],
    lm=lm_mini,
    skills=["data_exploration", "excel_modify"],
    max_turns=12,
    timeout=600.0,
)
result2 = rlm2.run()
print("turns:", result2.n_turns, "| tokens:", result2.total_prompt_tokens + result2.total_completion_tokens,
      "| seconds:", round(time.time() - t0, 1))
result2.payload

Spot-check the workbook. Row 3 should read AGO with a 60-month streak from
2021-M01 to 2025-M12, and the summary should count 163 countries analyzed and
97 with a streak:

In [ ]:
wb2 = load_workbook(REPORT2_PATH)
st = wb2["Streaks"]
{
    "sheets": wb2.sheetnames,
    "row3": [st.cell(row=3, column=c).value for c in range(1, 7)],
    "chart_present": len(st._charts) == 1,
    "cond_format_present": any("E3:E17" in str(r) for r in st.conditional_formatting),
    "summary": [wb2["Summary"]["B1"].value, wb2["Summary"]["B2"].value],
}

## Do the skills matter?

We ran this both ways. On the streaks task above, which spells out every
requirement, `skills=[]` also passes every check and costs less, because the
skill text rides along in every turn's prompt and the task already contains
the discipline the skills teach: inspect first, verify by reloading, never
leave placeholders.

The picture changes when the task is vague. We reran the streaks question
stripped of every presentation detail, just "produce a clear, professional
Excel report". With skills, the model got every number right and delivered a
tight report: a short summary, a streaks table, a small sample appendix.
Without skills, it shipped wrong numbers: 39 countries with a streak instead
of 97, and a 59-month longest streak instead of 60. Its own code shows why:
it detected runs with numpy's roll, which wraps the first month around to
the last, and the patch-up broke every streak touching the window edge. It
also dumped the entire filtered dataset into the workbook as a raw-data
sheet, 9,781 rows with all 24 SDMX plumbing columns. The skill playbooks
demand verify-before-submit and forbid raw dumps; the vague prompt gave the
no-skills run neither.

In short: on tightly specified tasks the skills are redundant cost; on vague
ones they fill in the taste your task text leaves out. They also lift pass
rates on bulk runs, as the SpreadsheetBench results in `CHANGELOG.md` show.

In [ ]:
t0 = time.time()
rlm3 = RLM.task(
    task=STREAKS_TASK,
    inputs={"data_file": File(DATA_PATH), "report_path": os.path.join(DATA_DIR, "streaks_noskills.xlsx")},
    outputs=["n_analyzed", "n_with_streak", "longest_country", "longest_len"],
    lm=lm_mini,
    skills=[],
    max_turns=12,
    timeout=600.0,
)
result3 = rlm3.run()
print("turns:", result3.n_turns, "| tokens:", result3.total_prompt_tokens + result3.total_completion_tokens,
      "| seconds:", round(time.time() - t0, 1))
result3.payload

## The takeaway

gpt-5.1 failed the first task not for lack of ability but because the data
was never in its context, and no model quality fixes missing data. gpt-5-mini
succeeded because fabric-rlm let it run code against all 1.57 million rows and
write a real, formatted workbook, verified cell by cell. In our runs:

| run | result | workbook | tokens | cost | seconds |
|---|---|---|---|---|---|
| direct call, gpt-5.1 | failed | none | 109,480 | $0.138 | 8.4 |
| fabric-rlm, gpt-5-mini | passed | correct, verified | 47,642 | $0.023 | 78.4 |

The direct call spent six times more failing than the small model spent
succeeding, and the same small model then cleared the streaks task, the chart,
and the conditional formatting for about four cents. When the answer
has to be computed from data that does not fit in context, an interpreter
beats a bigger model. The README section "When to use an RLM (and when not
to)" covers when this trade is worth it.